# Student ablation, trained and scored in one session (GPU)

Trains each arm of the ablation grid with the tuned values (trial 26 of the merged study,
`docs/tuning.md`) on the official split, then scores it on the gate (63,978), Civil bias
(21,577) and H2 (10,000) sets. Arms run in order of importance; an arm starts only if its
estimated time fits `BUDGET_H`, so the session never hits the quota or the 12-hour limit.

| Order | Arm | Est. minutes (last run) |
|---|---|---|
| 1 | `distilbert-mixed` (the tuned recipe) | 58 + 5 |
| 2 | `distilbert-gold` (no teacher: the leak check) | 58 + 5 |
| 3 | `distilbert-stack` (alpha 0) | 58 + 5 |
| 4 | `distilbert-mixed-5ep` (the edge check) | 73 + 5 |
| 5 | `minilm-mixed` | 47 + 5 |
| 6 | `modernbert-mixed` | 200 + 10 |

All six take about 9.5 hours; the first three about 3.3.

**Outputs** (`/kaggle/working/`, ids, scores and metrics only, no comment text):
`student/<arm>/` (model, `package/` with `model.onnx`, `predictions/{val,gate,bias_eval,h2_eval}.parquet`,
`predictions/thresholds.json`, `metrics.json`, `summary.json`, `scoring.json`),
`ablation_results.csv`, and `ablation_scored.zip` (the small files of every finished arm,
rebuilt after each arm).

**Before running:**
1. Accept the rules of both competitions (the **Data** tab, "I Understand and Accept"):
   `jigsaw-toxic-comment-classification-challenge` and `jigsaw-unintended-bias-in-toxicity-classification`.
2. **Add Input → Competitions**: add both.
3. **Add-ons → Secrets**: `GCS_TEACHER_KEY` (the service-account JSON), attached to this notebook;
   or `HF_TOKEN` with `EXTRA = ["storage.backend=hf"]`. Optional: `MLFLOW_TRACKING_URI`,
   `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD` (the team's MLflow server).
4. **Settings**: Accelerator **GPU T4 x2** or **T4** (one is used), Internet **On**.
5. Set `BUDGET_H` below to your GPU hours left minus half an hour (at most 11).
6. **Save Version → Save & Run All (Commit)**. Not the interactive session: its outputs are not kept.

In [ ]:
# Code version: pinned to the commit the sweep and the first ablation ran on (d07c861).
REPO_URL = "https://github.com/Nishaant-Soni/Distillate.git"
REPO_REF = "d07c861d3afe2137f5c9a4d2a60b2c5ec0023db2"

BUDGET_H = 11          # GPU hours this session may use: your quota left minus 0.5, at most 11

# Trial 26 of the merged study (28 trials): val mean AUC 0.99364 (docs/tuning.md).
TUNED = ["loss.alpha=0.3432987447660936", "loss.temperature=3.71295432376268",
         "train.lr=9.22542696771891e-05", "train.epochs=4", "model.max_length=256"]
GRID = {
    "distilbert-mixed":     [],
    "distilbert-gold":      ["loss.alpha=1.0"],
    "distilbert-stack":     ["loss.alpha=0.0"],
    "distilbert-mixed-5ep": ["train.epochs=5"],   # the check outside the study (tuning.md)
    "minilm-mixed":         ["model.name=minilm"],
    "modernbert-mixed":     ["model.name=modernbert"],
    "distilbert-tfidf":     ["soft_labels.arm=tfidf"],     # the attribution arms
    "distilbert-calqwen":   ["soft_labels.arm=calqwen"],
}
# Run order, most needed first. The attribution arms: add "distilbert-tfidf", "distilbert-calqwen".
ABLATION = ["distilbert-mixed", "distilbert-gold", "distilbert-stack", "distilbert-mixed-5ep",
            "minilm-mixed", "modernbert-mixed"]
# Minutes per arm in the first ablation run (training), plus scoring and the ONNX package.
EST_MIN = {"distilbert-mixed": 63, "distilbert-gold": 63, "distilbert-stack": 63,
           "distilbert-mixed-5ep": 78, "minilm-mixed": 52, "modernbert-mixed": 210,
           "distilbert-tfidf": 63, "distilbert-calqwen": 63}
NAMES = {"gate": "gate", "bias": "bias_eval", "h2": "h2_eval"}   # configs/gate.yaml stems (PR #33)
PACKAGE = True         # build each arm's §2.5 package (model.onnx, ...)
STORAGE = None         # no Optuna study in this notebook
EXTRA = []             # e.g. ["storage.backend=hf"] with an HF_TOKEN secret instead of GCS

import time
SESSION_START = time.perf_counter()

In [ ]:
import subprocess
import sys
from pathlib import Path

# iterative-stratification, mlflow, optuna, google-cloud-storage, and the ONNX export packages;
# torch and transformers come with the Kaggle image.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "iterative-stratification",
                "mlflow", "optuna==5.0.0", "google-cloud-storage", "onnx", "onnxscript",
                "onnxruntime"], check=True)

REPO_DIR = Path("/tmp/Distillate")   # outside /kaggle/working, so the repo is not saved as output
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--quiet", REPO_REF], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--quiet", "--ff-only"], check=False)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(REPO_DIR / "Model-Pipeline"))
print("code_commit:", CODE_COMMIT)

In [ ]:
import torch
import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "no GPU: set the accelerator to GPU T4"

In [ ]:
# Secrets, into the environment. Never printed. Whichever exist are loaded.
import os

from kaggle_secrets import UserSecretsClient

for name in ("GCS_TEACHER_KEY", "HF_TOKEN", "OPTUNA_STORAGE", "MLFLOW_TRACKING_URI",
             "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):   # the server's login
    try:
        os.environ[name] = UserSecretsClient().get_secret(name)
    except Exception:   # secret not attached
        pass
if STORAGE is None and os.environ.get("OPTUNA_STORAGE"):
    STORAGE = os.environ["OPTUNA_STORAGE"]
print("store key:", "GCS_TEACHER_KEY" in os.environ, "| shared study:", STORAGE is not None,
      "| MLflow server:", "MLFLOW_TRACKING_URI" in os.environ)

assert "GCS_TEACHER_KEY" in os.environ or "HF_TOKEN" in os.environ, \
    "attach the GCS_TEACHER_KEY secret (Add-ons -> Secrets): the soft-label arms need it"

## 1. Raw train (cleaned, in /tmp)

In [ ]:
from src.teacher.config import load_config as load_teacher_config
from src.teacher.data import check_frame, find_competition_dir, read_competition_split, to_raw_frame

DATA = Path("/tmp/wiki")
DATA.mkdir(parents=True, exist_ok=True)
RAW_TRAIN = DATA / "wiki_train_raw.parquet"

teacher_cfg = load_teacher_config()
comp_dir = find_competition_dir("/kaggle/input")   # fails if the competition is not attached
raw = to_raw_frame(read_competition_split(comp_dir, "train"))
print(check_frame(raw, teacher_cfg["data"]["expected_rows"]["train"]))   # 159,571 rows
raw.to_parquet(RAW_TRAIN, index=False)
del raw

In [ ]:
# Paths and data for every run: the official split by id; outputs under /kaggle/working.
OUT = Path("/kaggle/working")
BASE = [
    f"paths.raw_train={RAW_TRAIN}",
    f"paths.split_ids_dir={REPO_DIR / 'Data-Pipeline' / 'data' / 'splits'}",
    f"paths.out_dir={OUT / 'student'}",
    f"paths.mlflow_db={OUT / 'mlruns' / 'mlflow.db'}",
    f"paths.mlflow_artifacts={OUT / 'mlruns' / 'artifacts'}",
    f"paths.reports_dir={OUT / 'reports'}",
    "paths.cache_dir=/tmp/cache",
    "data.source=ids",
    *EXTRA,
]

## 2. Gate, bias and H2 texts (by the id lists in git) and the scorer

In [ ]:
import glob

import pandas as pd

from src.teacher.data import (clean_text, find_competition_dir, read_competition_split,
                              to_raw_frame)

SPLITS = REPO_DIR / "Data-Pipeline" / "data" / "splits"
SCORE_BATCH = 256


def ids(name):
    return [x.strip() for x in (SPLITS / f"{name}_ids.txt").read_text().splitlines() if x.strip()]


def take(frame, wanted, name):
    frame = frame.drop_duplicates("id").set_index("id")
    missing = [i for i in wanted if i not in frame.index]
    if missing:
        raise ValueError(f"{name}: {len(missing)} ids not in the source, e.g. {missing[:3]}")
    return frame.loc[wanted, ["text"]].reset_index()


def eval_texts():
    """The gate, bias and h2 texts by id (id lists in git), cleaned as in training."""
    wiki = to_raw_frame(read_competition_split(find_competition_dir("/kaggle/input"), "test"))
    out = {"gate": take(wiki[["id", "text"]], ids("gate"), "gate")}
    found = sorted(glob.glob("/kaggle/input/**/test_private_expanded.csv*", recursive=True))
    assert len(found) == 1, f"attach the unintended-bias competition, found {found}"
    civil = pd.read_csv(found[0], dtype=str, keep_default_na=False, usecols=["id", "comment_text"])
    civil = pd.DataFrame({"id": civil["id"].astype(str),
                          "text": civil["comment_text"].map(clean_text)})
    out["bias"] = take(civil, ids("bias_eval"), "bias")
    out["h2"] = take(civil, ids("h2_eval"), "h2")
    return out


# Loaded before the first arm trains, so a missing input fails early.
TEXTS = eval_texts()
print({k: len(v) for k, v in TEXTS.items()})   # gate 63,978, bias 21,577, h2 10,000

import json
import shutil
import time

import numpy as np
import torch

from src.student import train
from src.student.model import load_trained
from src.student.predict import check_prediction_frame, predict_logits, prediction_frame


def score(run_dir, device=None):
    """The gate, bias and h2 prediction files for one trained arm, next to its val.parquet."""
    run_dir = Path(run_dir)
    device = device or torch.device("cuda")
    meta = json.loads((run_dir / "model" / "meta.json").read_text())
    model, tokenizer = load_trained(run_dir / "model")
    model.to(device)
    rows = {}
    for key, frame in TEXTS.items():
        start = time.perf_counter()
        texts = frame["text"].tolist()
        order = np.argsort([len(x) for x in texts], kind="stable")   # less padding
        logits = predict_logits(model, tokenizer, [texts[i] for i in order],
                                int(meta["max_length"]), SCORE_BATCH, device, fp16=True)
        unsorted = np.empty_like(logits)
        unsorted[order] = logits
        pred = prediction_frame(frame["id"], unsorted, meta["model_version"])
        check_prediction_frame(pred)
        pred.to_parquet(run_dir / "predictions" / f"{NAMES[key]}.parquet", index=False)
        rows[NAMES[key]] = {"rows": len(pred), "seconds": round(time.perf_counter() - start, 1)}
    shutil.copy(run_dir / "model" / "thresholds.json", run_dir / "predictions" / "thresholds.json")
    (run_dir / "scoring.json").write_text(json.dumps(
        {"code_commit": CODE_COMMIT, "model_version": meta["model_version"], "files": rows},
        indent=2) + "\n")
    del model
    torch.cuda.empty_cache()
    return rows

## 3. Train and score each arm

In [ ]:
import zipfile

from src.student import train
from src.student.config import load_config
from src.teacher.config import LABELS

SMALL = ("summary.json", "metrics.json", "scoring.json", "meta.json", "thresholds.json",
         "thresholds_report.json")


def save_results():
    """ablation_results.csv and ablation_scored.zip for every arm finished so far."""
    table = pd.DataFrame({
        name: {"mean_auc": s["best_val_mean_auc"], "mean_pr_auc": s["best_val_mean_pr_auc"],
               **{f"auc_{k}": s["history"][s["best_epoch"]][f"val_auc_{k}"] for k in LABELS},
               "best_epoch": s["best_epoch"], "minutes": round(s["duration_s"] / 60, 1)}
        for name, s in runs.items()}).T
    table.to_csv(OUT / "ablation_results.csv")
    with zipfile.ZipFile(OUT / "ablation_scored.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted((OUT / "student").rglob("*")):
            rel = p.relative_to(OUT / "student")
            if p.is_file() and "package" not in rel.parts and (
                    p.name in SMALL or rel.parts[1:2] == ("predictions",)):
                z.write(p, rel)
        z.write(OUT / "ablation_results.csv", "ablation_results.csv")
        z.writestr("failed_and_skipped.json", json.dumps({"failed": failed, "skipped": skipped},
                                                          indent=2))
    return table


runs, failed, skipped = {}, {}, []
for name in ABLATION:
    used = (time.perf_counter() - SESSION_START) / 60
    if used + EST_MIN[name] * 1.1 > BUDGET_H * 60:
        skipped.append(name)
        print(f"{name}: skipped, {used:.0f} min used + about {EST_MIN[name]} would pass "
              f"BUDGET_H = {BUDGET_H}")
        continue
    print(f"\n=== {name} (at {used:.0f} min, about {EST_MIN[name]} min) ===")
    sets = BASE + TUNED + [f"run_name={name}", "mlflow.experiment=distillate-student-ablation",
                           f"package.enabled={str(PACKAGE).lower()}", *GRID[name]]
    try:     # one failed arm (out of memory, a download) doesn't lose the others
        runs[name] = train.run(load_config(None, sets))
        print(name, round(runs[name]["best_val_mean_auc"], 5),
              "| package:", runs[name].get("package", {}).get("status"))
        m = json.loads((Path(runs[name]["run_dir"]) / "metrics.json").read_text())
        print("  val PR-AUC per label:", json.dumps({k: round(m[k]["pr_auc"], 4) for k in m}))
        print("  scored:", score(runs[name]["run_dir"]))
    except Exception as e:
        runs.pop(name, None)
        failed[name] = f"{type(e).__name__}: {e}"
        print(name, "FAILED:", failed[name])
        torch.cuda.empty_cache()
    if runs:
        save_results()

print("\nfinished:", list(runs), "| failed:", failed or "none", "| skipped:", skipped or "none")
print(f"total {(time.perf_counter() - SESSION_START) / 3600:.2f} h")

## 4. Results (official validation split) and the output files

In [ ]:
if runs:
    print(save_results().round(4).to_string())
for p in sorted(OUT.rglob("*")):
    if p.is_file() and "mlruns" not in p.parts:
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.relative_to(OUT)}")